<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_13_Original_TRACE_Notebook_Recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import os, re, json, hashlib, ast, csv, zipfile, io, collections
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

MYDRIVE = Path('/content/drive/MyDrive')
PROJECT = MYDRIVE / 'NeuroFHIR_ICHI2027'
OUTPUT = PROJECT / 'phase13_outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
SHAREABLE_PATH = OUTPUT / 'ICHI2027_Phase13_SHAREABLE.json'
PRIVATE_PATH = OUTPUT / 'ICHI2027_Phase13_PRIVATE_MANIFEST.json'

assert MYDRIVE.is_dir(), 'Google Drive did not mount; cannot scan your existing notebooks.'

def sha256_file(p, limit=None):
    if limit is not None and p.stat().st_size > limit:
        return None
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_basename(p):
    name = Path(str(p).replace('\\','/')).name
    if len(name) > 160 or re.search(r'(?i)(?:RID|PTID|subject|participant)[_-]?\d{2,}', name):
        return '<WITHHELD_FILENAME>'
    if re.search(r'\d{8,}', name):
        return '<WITHHELD_FILENAME>'
    return name

def relative_private(p):
    return str(p.relative_to(MYDRIVE))


Mounted at /content/drive


In [2]:
phase12_candidates = [
    PROJECT / 'phase12_outputs' / 'ICHI2027_Phase12_SHAREABLE.json',
    Path('/content/ICHI2027_Phase12_SHAREABLE.json'),
]
phase12_file = next((p for p in phase12_candidates if p.is_file()), None)
if phase12_file is None:
    from google.colab import files
    upload = files.upload()
    candidates = [x for x in upload if x.lower().endswith('.json')]
    if len(candidates) != 1:
        raise FileNotFoundError('The Phase 12 shareable JSON is required.')
    phase12_file = Path('/content') / candidates[0]
    phase12_file.write_bytes(upload[candidates[0]])

phase12 = json.loads(phase12_file.read_text(encoding='utf-8'))
assert phase12.get('phase') == '12_original_source_and_TRACE_vocabulary_recovery'
previous_hash = sha256_file(phase12_file)
previous_known_notebooks = phase12.get('prior_trace_notebook_inventory', {})


In [3]:
ALL_FILES = []
walk_count = 0
truncated = False
SKIP_DIRS = {'.git', '.ipynb_checkpoints', 'node_modules', '__pycache__', '.venv', 'venv'}
MAX_ENTRIES = 50000
for current, dirs, names in os.walk(MYDRIVE):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
    for name in names:
        walk_count += 1
        if walk_count > MAX_ENTRIES:
            truncated = True
            break
        p = Path(current) / name
        if p.suffix.lower() in {'.ipynb', '.csv', '.tsv', '.parquet', '.json', '.zip'}:
            try:
                size = p.stat().st_size
                ALL_FILES.append({'path': p, 'size': size, 'ext': p.suffix.lower()})
            except OSError:
                pass
    if truncated:
        break

notebook_files = [x for x in ALL_FILES if x['ext'] == '.ipynb']
csv_files = [x for x in ALL_FILES if x['ext'] in {'.csv', '.tsv'}]
parquet_files = [x for x in ALL_FILES if x['ext'] == '.parquet']
zip_files = [x for x in ALL_FILES if x['ext'] == '.zip']
print({'files_examined': walk_count, 'notebooks_located': len(notebook_files),
       'csv_tsv_located': len(csv_files), 'zip_located': len(zip_files), 'truncated': truncated})


{'files_examined': 11820, 'notebooks_located': 281, 'csv_tsv_located': 1091, 'zip_located': 10, 'truncated': False}


In [4]:
SIGNALS = {
    'reliability_state': r'(?i)(?:TRACE[_ -]?STATE|RELIABILITY[_ -]?STATE|REVIEW_REQUIRED|\bCAUTION\b|\bTRUSTED\b)',
    'calibration_reference': r'(?i)(?:CALIBRATION[_ -]?(?:REF|ID|VERSION)|REPEAT[_ -]?(?:SCAN|ERROR)|\bQ9[05]\b|QUANTILE|THRESHOLD)',
    'rule_version': r'(?i)(?:TRACE[_ -]?RULE|RULE[_ -]?VERSION|METHOD[_ -]?VERSION|REFERENCE[_ -]?VERSION)',
    'image_pairing': r'(?i)(?:IMAGEUID|LONIUID|REPEAT[_ -]?PAIR|BL_EXAMDATE|M12_EXAMDATE)',
    'source_mri': r'(?i)(?:UCSFFSL51|ST29SV|ST88SV|LHIPPOC|RHIPPOC|FREESURFER)',
    'save_artifact': r'(?i)(?:\.to_csv\s*\(|\.to_parquet\s*\(|\.to_json\s*\(|\b(?:savez|save)\s*\()',
}
SIGNAL_PATTERNS = {k: re.compile(v) for k,v in SIGNALS.items()}
ALLOWED_EXT = {'.csv', '.parquet', '.json', '.zip'}


def source_text(cell):
    source = cell.get('source', '')
    return ''.join(source) if isinstance(source, list) else str(source)


def artifact_name(raw):
    s = str(raw).replace('\\', '/')
    candidate = Path(s).name
    if Path(candidate).suffix.lower() not in ALLOWED_EXT or len(candidate) > 160:
        return None
    if not re.fullmatch(r'[\w.() -]+', candidate):
        return None
    return safe_basename(candidate)


def strings_from_code(code):
    found = set()
    try:
        tree = ast.parse(code)
        for n in ast.walk(tree):
            if isinstance(n, ast.Constant) and isinstance(n.value, str):
                if len(n.value) < 300:
                    name = artifact_name(n.value)
                    if name:
                        found.add(name)
    except (SyntaxError, ValueError):
        pass
    for match in re.finditer(r'''[\"\']([^\"\'\n]{1,180}\.(?:csv|parquet|json|zip))[\"\']''', code, flags=re.I):
        name = artifact_name(match.group(1))
        if name:
            found.add(name)
    return sorted(found)


def inspect_nb(record):
    p = record['path']
    try:
        raw = p.read_bytes()
        obj = json.loads(raw)
        notebook_cells = obj.get('cells', [])
        if not isinstance(notebook_cells, list):
            return None
    except (OSError, ValueError):
        return None
    if len(raw) > 30 * 1024 * 1024:
        return None
    code = [c for c in notebook_cells if c.get('cell_type') == 'code']
    counters = collections.Counter()
    artifacts = set()
    candidate_indices = []
    embedded_outputs = 0
    for index, c in enumerate(notebook_cells):
        if c.get('cell_type') != 'code':
            continue
        src = source_text(c)
        hits = [k for k, pat in SIGNAL_PATTERNS.items() if pat.search(src)]
        counters.update(hits)
        artifacts.update(strings_from_code(src))
        if len(set(hits)) >= 2:
            candidate_indices.append(index)
        if c.get('outputs'):
            embedded_outputs += 1
    name = safe_basename(p.name)
    is_current_phase = name.lower().startswith(('ichi2027_colab_', 'copy_of_ichi2027_colab_'))
    priority = (5 * counters['reliability_state'] + 4 * counters['calibration_reference']
                + 4 * counters['rule_version'] + 2 * counters['image_pairing']
                + counters['source_mri'] + 2 * counters['save_artifact'])
    if re.search(r'(?i)(trace|neurofhir|repeat|landmark)', name):
        priority += 4
    return {
        'filename': name,
        'content_sha256': hashlib.sha256(raw).hexdigest(),
        'code_cell_count': len(code),
        'code_signal_cells': dict(counters),
        'candidate_cell_indices_zero_based': candidate_indices[:30],
        'referenced_artifact_basenames': sorted(artifacts)[:100],
        'embedded_outputs_cell_count': embedded_outputs,
        'search_priority_not_scientific_score': priority,
        'is_current_ichi_phase_notebook': is_current_phase,
        'private_relative_path': relative_private(p),
    }

profiles = []
skipped_oversize = 0
for rec in notebook_files:
    if rec['size'] > 30 * 1024 * 1024:
        skipped_oversize += 1
        continue
    p = inspect_nb(rec)
    if p is not None and not p['is_current_ichi_phase_notebook'] and p['search_priority_not_scientific_score'] > 0:
        profiles.append(p)
profiles.sort(key=lambda p: (-p['search_priority_not_scientific_score'], p['filename']))
print({'prior_notebook_candidates': len(profiles),
       'with_reliability_state_code': sum(p['code_signal_cells'].get('reliability_state',0)>0 for p in profiles),
       'skipped_oversize': skipped_oversize})


{'prior_notebook_candidates': 256, 'with_reliability_state_code': 65, 'skipped_oversize': 0}


In [5]:
# Search for files referenced by prior notebooks; names are evidence leads, not confirmation they were generated.
reference_index = collections.defaultdict(list)
for p in profiles:
    for basename in p['referenced_artifact_basenames']:
        reference_index[basename.lower()].append(p['content_sha256'][:16])

all_by_basename = collections.defaultdict(list)
for rec in ALL_FILES:
    all_by_basename[rec['path'].name.lower()].append(rec)

matched_artifacts_private = []
for name, notebook_hashes in sorted(reference_index.items()):
    matches = all_by_basename.get(name, [])
    if matches:
        for m in matches:
            matched_artifacts_private.append({
                'basename': safe_basename(m['path'].name),
                'relative_private_path': relative_private(m['path']),
                'bytes': m['size'],
                'mentioned_in_notebook_hash_prefixes': sorted(set(notebook_hashes)),
            })

print({'distinct_referenced_artifacts': len(reference_index),
       'existing_referenced_artifacts': len(matched_artifacts_private)})


{'distinct_referenced_artifacts': 1734, 'existing_referenced_artifacts': 1656}


In [6]:
def header_of_csv(path):
    with path.open('r', encoding='utf-8-sig', errors='replace', newline='') as f:
        first_line = f.readline(100000)
    if not first_line:
        return []
    delim = '\t' if first_line.count('\t') > first_line.count(',') else ','
    return [x.strip().strip('"') for x in next(csv.reader([first_line], delimiter=delim))]

RAW_FIELDS = {'ST29SV','ST88SV','ST24CV','ST83CV','ST40CV','ST99CV','ST10CV'}
STATE_RE = re.compile(r'(?i)(?:trace|reliab|policy).*(?:state|label|decision)|^(?:trace_state|reliability_state)$')
RULE_RE = re.compile(r'(?i)(?:trace|rule|method|policy).*(?:version|revision)')
REFERENCE_RE = re.compile(r'(?i)(?:calibration|threshold|repeat).*?(?:ref|id|version)|(?:ref|reference).*?(?:calibration|threshold)')
JOIN_RE = re.compile(r'(?i)^(RID|PTID|IMAGEUID|SOURCE_IMAGEUID|BL_IMAGEUID|M12_IMAGEUID|PERSON_ID)$')

header_profiles = []
csv_candidates = [x for x in csv_files if x['size'] <= 100 * 1024 * 1024]
for rec in csv_candidates[:2000]:
    p = rec['path']
    try:
        columns = header_of_csv(p)
    except (OSError, UnicodeError, csv.Error, StopIteration):
        continue
    raw = sorted(RAW_FIELDS.intersection({x.upper() for x in columns}))
    flags = {
        'state': any(STATE_RE.search(x) for x in columns),
        'rule_version': any(RULE_RE.search(x) for x in columns),
        'calibration_reference': any(REFERENCE_RE.search(x) for x in columns),
        'join_key': any(JOIN_RE.search(x) for x in columns),
    }
    if len(raw) >= 3 or flags['state'] or ('ucsffsl' in p.name.lower()):
        header_profiles.append({
            'filename': safe_basename(p.name),
            'private_relative_path': relative_private(p),
            'sha256': sha256_file(p, limit=100 * 1024 * 1024),
            'raw_ucsffsl_field_candidates_found': raw,
            'has_rid_join_candidate': any(x.upper() == 'RID' for x in columns),
            'trace_sidecar_column_candidates': flags,
            'has_full_trace_column_set': all(flags.values()),
            'column_count': len(columns),
        })

print({'csv_headers_inspected': min(len(csv_candidates), 2000),
       'possible_raw_or_trace_files': len(header_profiles),
       'trace_complete_header_candidates': sum(x['has_full_trace_column_set'] for x in header_profiles)})


{'csv_headers_inspected': 1091, 'possible_raw_or_trace_files': 0, 'trace_complete_header_candidates': 0}


In [7]:
full_vocabulary_zip = []
for rec in zip_files[:500]:
    p = rec['path']
    try:
        with zipfile.ZipFile(p) as z:
            names = {Path(x).name.upper() for x in z.namelist()}
        has_concept = 'CONCEPT.CSV' in names
        has_vocab = 'VOCABULARY.CSV' in names
        if has_concept or has_vocab:
            full_vocabulary_zip.append({
                'filename': safe_basename(p.name),
                'private_relative_path': relative_private(p),
                'bytes': rec['size'],
                'has_concept_csv': has_concept,
                'has_vocabulary_csv': has_vocab,
                'full_minimum_package_candidate': has_concept and has_vocab,
            })
    except (OSError, zipfile.BadZipFile, RuntimeError):
        pass
print({'vocabulary_zip_candidates': len(full_vocabulary_zip)})


{'vocabulary_zip_candidates': 0}


In [8]:
# Save only metadata. Prior notebook cells, output values, source identifiers and raw data are never exported.
notebook_public = [{k:v for k,v in p.items() if k != 'private_relative_path'} for p in profiles[:35]]
existing_files_public = [{k:v for k,v in p.items() if k != 'relative_private_path'} for p in matched_artifacts_private[:80]]
header_public = [{k:v for k,v in p.items() if k != 'private_relative_path'} for p in header_profiles[:80]]
zip_public = [{k:v for k,v in p.items() if k != 'private_relative_path'} for p in full_vocabulary_zip[:20]]

shareable = {
    'project': 'ICHI2027_neuroimaging_fhir_omop_fidelity',
    'phase': '13_original_TRACE_notebook_artifact_recovery',
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'result_status': 'PRIOR_NOTEBOOK_AND_ARTIFACT_LOCATOR_ONLY_NO_NEW_MICDM_LOAD_NO_COMPARATOR_SCORES',
    'privacy': 'Metadata only; no participant identifiers, private paths, notebook code, executed cell output or source rows.',
    'phase12_sha256': previous_hash,
    'phase12_prior_notebook_counts': previous_known_notebooks,
    'drive_inventory': {
        'files_scanned': walk_count,
        'truncated': truncated,
        'notebooks_present': len(notebook_files),
        'csv_tsv_present': len(csv_files),
        'zip_files_present': len(zip_files),
        'prior_notebooks_with_relevant_code': len(profiles),
        'prior_notebooks_with_reliability_state_code': sum(bool(p['code_signal_cells'].get('reliability_state')) for p in profiles),
        'prior_notebooks_with_calibration_code': sum(bool(p['code_signal_cells'].get('calibration_reference')) for p in profiles),
        'notebooks_skipped_oversize': skipped_oversize,
    },
    'prior_notebook_candidates': notebook_public,
    'referenced_artifact_names_on_drive': existing_files_public,
    'possible_raw_or_trace_csv_headers': header_public,
    'possible_athena_archives': zip_public,
    'gates': {
        'actual_prior_notebooks_inspected': len(profiles) > 0,
        'complete_trace_sidecar_file_candidate_found': any(x['has_full_trace_column_set'] for x in header_profiles),
        'raw_ucsffsl_export_header_candidate_found': any(len(x['raw_ucsffsl_field_candidates_found'])>=5 and x['has_rid_join_candidate'] for x in header_profiles),
        'athena_full_package_candidate_found': any(x['full_minimum_package_candidate'] for x in full_vocabulary_zip),
        'original_TRACE_state_rule_calibration_provenance_verified': False,
        'original_raw_source_values_reconciled': False,
        'source_units_approved': False,
        'DICOM_study_and_series_UIDs_verified': False,
        'OMOP_vocabulary_mapping_approved': False,
        'official_MICDM_load_completed': False,
        'genuine_four_arm_comparison_completed': False,
    },
    'comparison_results': None,
    'next_gate': 'REVIEW_TOP_PRIOR_NOTEBOOK_CODE_AND_EXISTING_ARTIFACT_CANDIDATES_IF_FOUND; OBTAIN_ORIGINAL_ADNI_EXPORT_AND_ATHENA_PACKAGE',
}

private = {
    'run_utc': shareable['run_utc'],
    'private_note': 'Private file locations; do not upload this manifest, raw source data or old notebook outputs to public GitHub.',
    'notebook_candidates': profiles,
    'referenced_artifacts': matched_artifacts_private,
    'raw_or_trace_header_candidates': header_profiles,
    'athena_archive_candidates': full_vocabulary_zip,
}
SHAREABLE_PATH.write_text(json.dumps(shareable, indent=2, ensure_ascii=False), encoding='utf-8')
PRIVATE_PATH.write_text(json.dumps(private, indent=2, ensure_ascii=False), encoding='utf-8')
print({'phase13_json_saved': SHAREABLE_PATH.name,
       'prior_notebook_candidates': len(profiles),
       'trace_column_set_candidates': sum(x['has_full_trace_column_set'] for x in header_profiles),
       'full_vocab_zip_candidates': sum(x['full_minimum_package_candidate'] for x in full_vocabulary_zip)})


{'phase13_json_saved': 'ICHI2027_Phase13_SHAREABLE.json', 'prior_notebook_candidates': 256, 'trace_column_set_candidates': 0, 'full_vocab_zip_candidates': 0}


In [9]:
from google.colab import files
files.download(str(SHAREABLE_PATH))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>